In [14]:
from datetime import datetime, timedelta

PKT_FMT = "%Y-%m-%d %H:%M:%S"


def detect_port_scan(packets, port_threshold=10, window_seconds=30):
    """
    Detect (src_ip -> dst_ip) pairs that contact at least
    'port_threshold' distinct destination ports within
    'window_seconds'.
    """

    # Sort packets by timestamp
    packets = sorted(
        packets,
        key=lambda p: datetime.strptime(p["timestamp"], PKT_FMT)
    )

    # Group packets by (source IP, destination IP)
    by_pair = {}

    for p in packets:
        key = (p["src_ip"], p["dst_ip"])
        by_pair.setdefault(key, []).append(p)

    results = {}

    # Analyze each source-destination pair
    for key, pkts in by_pair.items():

        for i in range(len(pkts)):
            start = datetime.strptime(pkts[i]["timestamp"], PKT_FMT)
            end = start + timedelta(seconds=window_seconds)

            ports_in_window = {
                p["dst_port"]
                for p in pkts
                if start <= datetime.strptime(p["timestamp"], PKT_FMT) <= end
            }

            if len(ports_in_window) >= port_threshold:
                results[key] = {
                    "distinct_ports": len(ports_in_window),
                    "ports": sorted(ports_in_window),
                }
                break

    return results

In [15]:
packets = [
    {"timestamp": "2026-08-04 10:00:00", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 20},
    {"timestamp": "2026-08-04 10:00:02", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 21},
    {"timestamp": "2026-08-04 10:00:04", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 22},
    {"timestamp": "2026-08-04 10:00:06", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 23},
    {"timestamp": "2026-08-04 10:00:08", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 24},
    {"timestamp": "2026-08-04 10:00:10", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 25},
    {"timestamp": "2026-08-04 10:00:12", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 26},
    {"timestamp": "2026-08-04 10:00:14", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 27},
    {"timestamp": "2026-08-04 10:00:16", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 28},
    {"timestamp": "2026-08-04 10:00:18", "src_ip": "192.168.1.10", "dst_ip": "10.0.0.5", "dst_port": 29},
]

print(detect_port_scan(packets))

{('192.168.1.10', '10.0.0.5'): {'distinct_ports': 10, 'ports': [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]}}
